# Multimodal Models

Yeh notebook [AI for Beginners Curriculum](http://aka.ms/ai-beginners) se li gayi hai.

## CLIP ke saath experiment karna

[CLIP](https://openai.com/blog/clip/) ko OpenAI ne publicly release kiya hai, isliye aap ise alag-alag tasks ke liye use karke dekh sakte hain, jisme zero-shot image classification bhi shaamil hai. Dhyan rakhiye ki yeh resources ke maamle mein kaafi demanding hai!

In [ ]:
import sys
!{sys.executable} -m pip install git+https://github.com/openai/CLIP.git

Hum pehle yeh make sure karenge ki agar GPU available hai toh hum use kar sakein, aur phir CLIP model load karenge.

In [ ]:
import torch
import clip
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
np.set_printoptions(precision=2,suppress=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

Chalo [Oxford-IIIT dataset](https://www.robots.ox.ac.uk/~vgg/data/pets/) se billiyon ki images ka ek subset bhi le lete hain:

In [ ]:
!wget https://mslearntensorflowlp.blob.core.windows.net/data/oxcats.tar.gz
!tar xfz oxcats.tar.gz
!rm oxcats.tar.gz

### Zero-shot Image Classification

CLIP ka main kaam yeh hai ki woh kisi image ko text prompt ke saath match kar sakta hai. Jaise agar hamare paas ek billi ki image hai aur hum use text prompts "a cat", "a penguin", "a bear" ke saath match karne ki koshish karte hain, toh pehla option sabse zyada probable hoga. Isse hum conclude kar sakte hain ki hum ek billi ke saath kaam kar rahe hain. Hamein model ko train karne ki zaroorat nahi hai kyunki woh pehle hi ek bade dataset par pre-train ho chuka hai - isliye ise **zero-shot** kehte hain.

In [ ]:
image = preprocess(Image.open("oxcats/Maine_Coon_1.jpg")).unsqueeze(0).to(device)
text = clip.tokenize(["a penguin", "a bear", "a cat"]).to(device)

with torch.no_grad():
    image_features = model.encode_image(image)
    text_features = model.encode_text(text)
    
    logits_per_image, logits_per_text = model(image, text)
    probs = logits_per_image.softmax(dim=-1).cpu().numpy()

print("Label probs:", probs)

### Intelligent Image Search

Pichhle example mein ek image aur teen text prompts the. Hum CLIP ko ek alag context mein bhi use kar sakte hain, jaise hum billi ki kai images le sakte hain, aur phir woh image choose kar sakte hain jo text description se sabse zyada match karti ho:

In [ ]:
cats_img = [ Image.open(os.path.join("oxcats",x)) for x in os.listdir("oxcats") ] 
cats = torch.cat([ preprocess(i).unsqueeze(0) for i in cats_img ]).to(device)
text = clip.tokenize(["a very fat gray cat"]).to(device)
with torch.no_grad():
    logits_per_image, logits_per_text = model(cats, text)
    res = logits_per_text.softmax(dim=-1).argmax().cpu().numpy()

print("Img Index:", res)

plt.imshow(cats_img[res])

## Key Takeaways

Pre-trained CLIP model ko general objects par image classification jaise tasks ke liye bina kisi domain-specific training ke use kiya ja sakta hai. Saath hi, yeh zyada flexible classification/image search ki permission deta hai, jisme picture mein objects ke spatial structure ko bhi dhyan mein rakha jaata hai.

CLIP ke ek aur exciting use ke liye, **VQGAN+CLIP** dekhiye.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->